# Módulo de Configuración Histórica
## Notebook 00 — Tablas BSC Dimensiones

Este notebook crea y carga las cuatro tablas de dimensiones con prefijo `bsc_`:

| Tabla | Descripción |
|---|---|
| `bsc_dim_usuario` | Personas registradas en el sistema |
| `bsc_dim_rol` | Roles o posiciones dentro del modelo comercial |
| `bsc_dim_evaluacion` | Indicadores o métricas que se miden |
| `bsc_dim_esquema` | Esquemas de compensación disponibles |

> **Prerrequisito**: Asegúrate de que este notebook esté adjunto a un Lakehouse antes de ejecutarlo.  
> En la barra lateral de Fabric selecciona **Add Lakehouse** y elige tu Lakehouse de destino.

---
## 0 · Configuración del Lakehouse

In [ ]:
# ─── Ajusta este valor al nombre de tu Lakehouse en Fabric ───────────────────
LAKEHOUSE_NAME = "BI - Bandelta LH"
# ─────────────────────────────────────────────────────────────────────────────

spark.sql(f"USE `{LAKEHOUSE_NAME}`")
print(f"✅ Usando Lakehouse: {LAKEHOUSE_NAME}")

---
## 1 · bsc_dim_usuario

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS bsc_dim_usuario (
    id_usuario      INT           NOT NULL  COMMENT 'Llave primaria del usuario',
    nombre          STRING        NOT NULL  COMMENT 'Nombre completo',
    email           STRING        NOT NULL  COMMENT 'Correo electrónico corporativo (único)',
    codigo_empleado STRING                  COMMENT 'Número de empleado en el sistema RH',
    area            STRING                  COMMENT 'Área o departamento al que pertenece',
    activo          BOOLEAN       NOT NULL  COMMENT 'Indica si el usuario está activo',
    fecha_creacion  TIMESTAMP     NOT NULL  COMMENT 'Timestamp de alta en el sistema',
    fecha_baja      TIMESTAMP               COMMENT 'Timestamp de baja; NULL si sigue activo'
)
USING DELTA
COMMENT 'Catálogo de usuarios del módulo de compensación histórica'
TBLPROPERTIES (
    'delta.minReaderVersion' = '1',
    'delta.minWriterVersion' = '2'
)
""")

print("✅ Tabla bsc_dim_usuario creada (o ya existía).")

In [ ]:
spark.sql("""
INSERT INTO bsc_dim_usuario
    (id_usuario, nombre,          email,                    codigo_empleado, area,       activo, fecha_creacion,      fecha_baja)
VALUES
    -- ── Agrega una fila por usuario ─────────────────────────────────────────
    (1,          'Nombre Ejemplo', 'usuario@empresa.com',   'EMP-001',       'Ventas',   true,   current_timestamp(), NULL)
    -- (2,       '...',            '...',                   '...',           '...',      true,   current_timestamp(), NULL)
""")

spark.sql("SELECT * FROM bsc_dim_usuario ORDER BY id_usuario").show(truncate=False)

---
## 2 · bsc_dim_rol

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS bsc_dim_rol (
    id_rol       INT     NOT NULL  COMMENT 'Llave primaria del rol',
    nombre_rol   STRING  NOT NULL  COMMENT 'Nombre del rol, p.ej. Asesor Comercial',
    descripcion  STRING            COMMENT 'Descripción detallada del rol',
    nivel        STRING            COMMENT 'Nivel jerárquico: Operativo, Táctico, Estratégico',
    activo       BOOLEAN NOT NULL  COMMENT 'Indica si el rol está vigente'
)
USING DELTA
COMMENT 'Catálogo de roles del modelo de compensación'
TBLPROPERTIES (
    'delta.minReaderVersion' = '1',
    'delta.minWriterVersion' = '2'
)
""")

print("✅ Tabla bsc_dim_rol creada (o ya existía).")

In [ ]:
spark.sql("""
INSERT INTO bsc_dim_rol
    (id_rol, nombre_rol,          descripcion,                          nivel,        activo)
VALUES
    -- ── Agrega una fila por rol ──────────────────────────────────────────────
    (1,      'Asesor Comercial',  'Venta directa a cliente final',      'Operativo',  true)
    -- (2,   '...',              '...',                                 '...',        true)
""")

spark.sql("SELECT * FROM bsc_dim_rol ORDER BY id_rol").show(truncate=False)

---
## 3 · bsc_dim_evaluacion

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS bsc_dim_evaluacion (
    id_evaluacion      INT     NOT NULL  COMMENT 'Llave primaria de la evaluación',
    nombre_evaluacion  STRING  NOT NULL  COMMENT 'Nombre del indicador, p.ej. Ventas Netas',
    descripcion        STRING            COMMENT 'Qué mide y cómo se calcula',
    tipo_metrica       STRING  NOT NULL  COMMENT 'porcentaje | valor_absoluto | ratio | conteo',
    unidad_medida      STRING            COMMENT 'MXN, unidades, %, clientes, etc.',
    activo             BOOLEAN NOT NULL  COMMENT 'Indica si la evaluación está vigente'
)
USING DELTA
COMMENT 'Catálogo de evaluaciones/indicadores del modelo de compensación'
TBLPROPERTIES (
    'delta.minReaderVersion' = '1',
    'delta.minWriterVersion' = '2'
)
""")

print("✅ Tabla bsc_dim_evaluacion creada (o ya existía).")

In [ ]:
spark.sql("""
INSERT INTO bsc_dim_evaluacion
    (id_evaluacion, nombre_evaluacion, descripcion,                  tipo_metrica,      unidad_medida, activo)
VALUES
    -- ── Agrega una fila por indicador ───────────────────────────────────────
    (1,             'Ventas Netas',    'Monto total de ventas netas', 'valor_absoluto',  'MXN',         true)
    -- (2,          '...',            '...',                          '...',             '...',         true)
""")

spark.sql("SELECT * FROM bsc_dim_evaluacion ORDER BY id_evaluacion").show(truncate=False)

---
## 4 · bsc_dim_esquema

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS bsc_dim_esquema (
    id_esquema      INT     NOT NULL  COMMENT 'Llave primaria del esquema',
    nombre_esquema  STRING  NOT NULL  COMMENT 'Nombre descriptivo, p.ej. Esquema Ventas 2024',
    descripcion     STRING            COMMENT 'Objetivo y alcance del esquema',
    tipo_esquema    STRING            COMMENT 'individual | grupal | mixto',
    activo          BOOLEAN NOT NULL  COMMENT 'Indica si el esquema está vigente'
)
USING DELTA
COMMENT 'Catálogo de esquemas de compensación'
TBLPROPERTIES (
    'delta.minReaderVersion' = '1',
    'delta.minWriterVersion' = '2'
)
""")

print("✅ Tabla bsc_dim_esquema creada (o ya existía).")

In [ ]:
spark.sql("""
INSERT INTO bsc_dim_esquema
    (id_esquema, nombre_esquema,          descripcion,                              tipo_esquema,  activo)
VALUES
    -- ── Agrega una fila por esquema ─────────────────────────────────────────
    (1,          'Esquema Ventas 2024',   'Compensación por volumen de ventas',     'individual',  true)
    -- (2,       '...',                  '...',                                    '...',         true)
""")

spark.sql("SELECT * FROM bsc_dim_esquema ORDER BY id_esquema").show(truncate=False)

---
## 5 · Resumen de tablas

In [ ]:
tablas_bsc = ["bsc_dim_usuario", "bsc_dim_rol", "bsc_dim_evaluacion", "bsc_dim_esquema"]

print(f"{'TABLA':<25} {'REGISTROS':>10}")
print("-" * 37)
for tabla in tablas_bsc:
    cnt = spark.sql(f"SELECT COUNT(*) as c FROM {tabla}").collect()[0].c
    print(f"{tabla:<25} {cnt:>10,}")

print("\n✅ Notebook 00 completado. Continúa con 01_dimensiones_config_historico.ipynb")